# 1.Criação das camadas do projeto e ingestão bronze

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS mvp_data_engineering;

CREATE SCHEMA IF NOT EXISTS mvp_data_engineering.bronze;
CREATE SCHEMA IF NOT EXISTS mvp_data_engineering.silver;
CREATE SCHEMA IF NOT EXISTS mvp_data_engineering.gold;


In [0]:
%sql

SHOW SCHEMAS IN mvp_data_engineering;


databaseName
bronze
default
gold
information_schema
silver


##1.1 Ingestão dos arquivos via Catálago do Databricks

In [0]:
# Criando os paths para criação da camada bronze
pokemon_path = "/Volumes/mvp_data_engineering/bronze/raw_files/pokemon.csv"
combats_path = "/Volumes/mvp_data_engineering/bronze/raw_files/combats.csv"
tests_path = "/Volumes/mvp_data_engineering/bronze/raw_files/tests.csv"

print(f"pokemon_path: {pokemon_path}")
print(f"combats_path: {combats_path}")
print(f"tests_path: {tests_path}")

pokemon_path: /Volumes/mvp_data_engineering/bronze/raw_files/pokemon.csv
combats_path: /Volumes/mvp_data_engineering/bronze/raw_files/combats.csv
tests_path: /Volumes/mvp_data_engineering/bronze/raw_files/tests.csv


In [0]:
#Leitura preliminar dos dados
pokemon_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(pokemon_path)
)

combats_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(combats_path)
)

tests_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(tests_path)
)

display(pokemon_raw.limit(10))
display(combats_raw.limit(10))
display(tests_raw.limit(10))

#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,false
2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,false
3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,false
4,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,false
5,Charmander,Fire,null,39,52,43,60,50,65,1,false
6,Charmeleon,Fire,null,58,64,58,80,65,80,1,false
7,Charizard,Fire,Flying,78,84,78,109,85,100,1,false
8,Mega Charizard X,Fire,Dragon,78,130,111,130,85,100,1,false
9,Mega Charizard Y,Fire,Flying,78,104,78,159,115,100,1,false
10,Squirtle,Water,null,44,48,65,50,64,43,1,false


First_pokemon,Second_pokemon,Winner
266,298,298
702,701,701
191,668,668
237,683,683
151,231,151
657,752,657
192,134,134
73,545,545
220,763,763
302,31,31


First_pokemon,Second_pokemon
129,117
660,211
706,115
195,618
27,656
126,222
436,207
121,36
169,636
302,688


In [0]:
#Verificando os schemas iniciais
pokemon_raw.printSchema()
combats_raw.printSchema()
tests_raw.printSchema()

root
 |-- #: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Type 1: string (nullable = true)
 |-- Type 2: string (nullable = true)
 |-- HP: integer (nullable = true)
 |-- Attack: integer (nullable = true)
 |-- Defense: integer (nullable = true)
 |-- Sp. Atk: integer (nullable = true)
 |-- Sp. Def: integer (nullable = true)
 |-- Speed: integer (nullable = true)
 |-- Generation: integer (nullable = true)
 |-- Legendary: boolean (nullable = true)

root
 |-- First_pokemon: integer (nullable = true)
 |-- Second_pokemon: integer (nullable = true)
 |-- Winner: integer (nullable = true)

root
 |-- First_pokemon: integer (nullable = true)
 |-- Second_pokemon: integer (nullable = true)



##1.2 Criação de metadados e persistência em Delta

In [0]:
#Adicionando colunas de ingestão e origem
from pyspark.sql.functions import current_timestamp, lit

pokemon_bronze = (
    pokemon_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("pokemon.csv"))
)

combats_bronze = (
    combats_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("combats.csv"))
)

tests_bronze = (
    tests_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("tests.csv"))
)

In [0]:
#Persistir a camada bronze em Delta
(
    pokemon_bronze.write
    .format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.bronze.pokemon_raw")
)

(
    combats_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.bronze.combats_raw")
)

(
    tests_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.bronze.tests_raw")
)

In [0]:
%sql

SHOW TABLES IN mvp_data_engineering.bronze;

database,tableName,isTemporary
bronze,combats_raw,false
bronze,pokemon_raw,false
bronze,tests_raw,false


In [0]:
%sql

SELECT *
FROM mvp_data_engineering.bronze.pokemon_raw
LIMIT 10;

#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,_ingestion_timestamp,_source_file
1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
4,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
5,Charmander,Fire,null,39,52,43,60,50,65,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
6,Charmeleon,Fire,null,58,64,58,80,65,80,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
7,Charizard,Fire,Flying,78,84,78,109,85,100,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
8,Mega Charizard X,Fire,Dragon,78,130,111,130,85,100,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
9,Mega Charizard Y,Fire,Flying,78,104,78,159,115,100,1,false,2026-09-27T00:43:17.752Z,pokemon.csv
10,Squirtle,Water,null,44,48,65,50,64,43,1,false,2026-09-27T00:43:17.752Z,pokemon.csv


#2. Análise preliminar de qualidade dos dados

Nesta etapa são avaliados os dados presentes na camada Bronze antes da
aplicação das regras de tratamento e padronização da camada Silver.

In [0]:
datasets = {
    "pokemon": pokemon_bronze,
    "combats": combats_bronze,
    "tests": tests_bronze
}

for name, df in datasets.items():
    print(
        f"{name}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

pokemon: 800 registros | 14 colunas
combats: 50000 registros | 5 colunas
tests: 10000 registros | 4 colunas


##2.1 Verificação de Nulos

In [0]:
#Criando uma função para verificação de valores nulos
from pyspark.sql.functions import col, count, when

def null_analysis(df):
    return df.select([
        count(
            when(col(f"`{c}`").isNull(), 1)
        ).alias(c)
        for c in df.columns
    ])

In [0]:
display(null_analysis(pokemon_bronze))

#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,_ingestion_timestamp,_source_file
0,1,0,386,0,0,0,0,0,0,0,0,0,0


In [0]:
display(null_analysis(combats_bronze))

First_pokemon,Second_pokemon,Winner,_ingestion_timestamp,_source_file
0,0,0,0,0


In [0]:
display(null_analysis(tests_bronze))

First_pokemon,Second_pokemon,_ingestion_timestamp,_source_file
0,0,0,0


Valores nulos encontrados na tabela **pokemon_bronze** no campo **Type 2**. Isso é esperado, visto que temos Pokémon que podem ou não ter mais de um tipo.

##2.2 Verificação de duplicatas

In [0]:
#Criando uma função para verificação de duplicidades
from pyspark.sql.functions import count

display(
    pokemon_bronze
    .groupBy("#")
    .count()
    .filter(col("count") > 1)
)

#,count


In [0]:
total_pokemon = pokemon_bronze.count()
unique_pokemon = pokemon_bronze.dropDuplicates().count()

print("Total:", total_pokemon)
print("Únicos:", unique_pokemon)
print("Duplicados:", total_pokemon - unique_pokemon)

Total: 800
Únicos: 800
Duplicados: 0


In [0]:
total_combats = combats_bronze.count()
unique_combats = combats_bronze.dropDuplicates().count()

print("Total:", total_combats)
print("Únicos:", unique_combats)
print("Duplicados:", total_combats - unique_combats)

Total: 50000
Únicos: 48048
Duplicados: 1952


Pode-se concluir que existem chaves de identificações distintas para cada Pokémon. As chaves podem se repetir em **combats_bronze**, pois pokemons de mesma "espécie" podem se enfrentar ou um mesmo Pokémon pode batalhar diversas vezes com o mesmo oponente.

##2.3 Verificação da integridade das informações

Etapa para identificar se os dados das colunas estão padronizados com a mesma formatação.

In [0]:
display(
    pokemon_bronze
    .select("Type 1")
    .distinct()
    .orderBy("Type 1")
)

Type 1
Bug
Dark
Dragon
Electric
Fairy
Fighting
Fire
Flying
Ghost
Grass


In [0]:
display(
    pokemon_bronze
    .select("Type 2")
    .distinct()
    .orderBy("Type 2")
)

Type 2
null
Bug
Dark
Dragon
Electric
Fairy
Fighting
Fire
Flying
Ghost


In [0]:
display(
    pokemon_bronze
    .groupBy("Generation")
    .count()
    .orderBy("Generation")
)

Generation,count
1,166
2,106
3,160
4,121
5,165
6,82


In [0]:
display(
    pokemon_bronze
    .groupBy("Legendary")
    .count()
)

Legendary,count
true,65
false,735


Feita a verificação da integridade dos dados, podemos concluir que as tipagens estão corretas e que a indicação de lendário está consistente.

##2.4 Verificação dos atributos

Etapa para identificar possíveis outliers preliminares.

In [0]:
#Verificando a acurácia dos atributos
from pyspark.sql.functions import col

# Colunas numéricas que queremos analisar
stats_columns = [
    "HP",
    "Attack",
    "Defense",
    "Sp. Atk",
    "Sp. Def",
    "Speed"
]

# DataFrame temporário somente para análise estatística
pokemon_stats = pokemon_bronze.select(
    col("`HP`").alias("hp"),
    col("`Attack`").alias("attack"),
    col("`Defense`").alias("defense"),
    col("`Sp. Atk`").alias("sp_attack"),
    col("`Sp. Def`").alias("sp_defense"),
    col("`Speed`").alias("speed")
)

display(pokemon_stats.summary())

summary,hp,attack,defense,sp_attack,sp_defense,speed
count,800,800,800,800,800,800
mean,69.25875,79.00125,73.8425,72.82,71.9025,68.2775
stddev,25.534669032332047,32.45736586949843,31.183500559332927,32.72229416880157,27.82891579711745,29.06047371716149
min,1,5,5,10,20,5
25%,50,55,50,49,50,45
50%,65,75,70,65,70,65
75%,80,100,90,95,90,90
max,255,190,230,194,230,180


In [0]:
#Verificação de valores negativos
stats_columns_clean = [
    "hp",
    "attack",
    "defense",
    "sp_attack",
    "sp_defense",
    "speed"
]

for c in stats_columns_clean:
    invalid = pokemon_stats.filter(
        col(c) < 0
    ).count()

    print(f"{c}: {invalid} valores negativos")

hp: 0 valores negativos
attack: 0 valores negativos
defense: 0 valores negativos
sp_attack: 0 valores negativos
sp_defense: 0 valores negativos
speed: 0 valores negativos


Pode-se verificar que não há valores de status negativos e sem outliers identifcados.

##2.5 Verificando integridade das chaves referenciais

In [0]:
#Verificando a integridade referencial das batalhas
pokemon_ids = pokemon_bronze.select(
    col("#").alias("pokemon_id")
)

invalid_first = (
    combats_bronze
    .join(
        pokemon_ids,
        combats_bronze["First_pokemon"] == pokemon_ids["pokemon_id"],
        "left_anti"
    )
)

print("First_pokemon inválidos:", invalid_first.count())

invalid_second = (
    combats_bronze
    .join(
        pokemon_ids,
        combats_bronze["Second_pokemon"] == pokemon_ids["pokemon_id"],
        "left_anti"
    )
)

print("Second_pokemon inválidos:", invalid_second.count())

invalid_winner = (
    combats_bronze
    .join(
        pokemon_ids,
        combats_bronze["Winner"] == pokemon_ids["pokemon_id"],
        "left_anti"
    )
)

print("Winner inválidos:", invalid_winner.count())

First_pokemon inválidos: 0
Second_pokemon inválidos: 0
Winner inválidos: 0


In [0]:
#Verificando se o vencedor sempre será o Pokémon 1 ou Pokémon 2
invalid_winners = combats_bronze.filter(
    (col("Winner") != col("First_pokemon")) &
    (col("Winner") != col("Second_pokemon"))
)

print(
    "Batalhas com vencedor inválido:",
    invalid_winners.count()
)

Batalhas com vencedor inválido: 0


In [0]:
self_battles = combats_bronze.filter(
    col("First_pokemon") == col("Second_pokemon")
)

print(
    "Batalhas contra o próprio Pokémon:",
    self_battles.count()
)

Batalhas contra o próprio Pokémon: 0


#3. Resumo da análise preliminar

In [0]:
quality_summary = [
    ("pokemon", "total_records", pokemon_bronze.count()),
    ("combats", "total_records", combats_bronze.count()),
    ("tests", "total_records", tests_bronze.count()),
    ("pokemon", "duplicate_ids",
        pokemon_bronze.count()
        - pokemon_bronze.dropDuplicates(["#"]).count()),
    ("combats", "invalid_winner",
        invalid_winners.count()),
    ("combats", "self_battles",
        self_battles.count()),
    ("combats", "invalid_first_pokemon",
        invalid_first.count()),
    ("combats", "invalid_second_pokemon",
        invalid_second.count()),
    ("combats", "invalid_winner_id",
        invalid_winner.count())
]

quality_df = spark.createDataFrame(
    quality_summary,
    ["dataset", "check", "value"]
)

print("Dados de qualidade gerados")
display(quality_df)

Dados de qualidade gerados


dataset,check,value
pokemon,total_records,800
combats,total_records,50000
tests,total_records,10000
pokemon,duplicate_ids,0
combats,invalid_winner,0
combats,self_battles,0
combats,invalid_first_pokemon,0
combats,invalid_second_pokemon,0
combats,invalid_winner_id,0
